# ChaoSSM — Phase 14: Regime-pure reservoir controls for the revision

**Purpose.** This notebook answers the two major concerns of Reviewer 1 and the Lyapunov-on-forecast
question of Reviewer 2, for the headline logistic-map fair test, by *extending* the validated Phase 10
reservoir harness with three additions and changing nothing else:

1. **An `ordered` regime (lambda ~= -0.30)** added to the SAME ChaosCell, giving a three-point
   dynamical-regime axis: ordered (-0.30) / edge (-0.02) / deep (+0.30). Because only the calibrated
   exponent differs between these arms (identical cell, coupling, gate-removed reservoir setup,
   gamma-calibration machinery, task, protocol, seeds), any performance difference is attributable to
   the dynamical regime and not to the architecture. This is the regime-pure analogue, within the
   reservoir setting, of the trained-setting periodic-regime ablation (manuscript Table 3), and it
   directly addresses R1 major concern 1 for the positive result.

2. **Linear short-term memory capacity (MC) and a separation / kernel-quality measure**, computed on
   the frozen reservoir states for each regime. These are the canonical reservoir-computing diagnostics
   (Jaeger 2001; Dambre et al. 2012; Legenstein & Maass 2007) that connect the Lyapunov exponent to task
   performance. Whether or not MC and separation peak near the edge, reporting them addresses R1 major concern 2
   (intermediate computational properties) with measurement rather than assertion.

3. **The conditional (input-driven) Lyapunov exponent measured on the forecast task**, for the frozen
   edge reservoir under random-B vs trained-B. This tests the synchronization mechanism directly on
   forecasting (R2 comment 3b) rather than inferring it from copy.

The core cells (setup, config, ChaosCell/LinearCell/GRU, gamma-calibration, SeqModel, tasks, trainer)
are reproduced **verbatim** from the validated Phase 10 n=20 notebook; the only change to Config is the
added `ordered` lambda target. New code is clearly sectioned below.


## Setup (verbatim from Phase 10)

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase14_regime_controls"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
# NOTE: this notebook writes ONLY to its own results/phase14_regime_controls/ folder.
# It creates new phase14_*.csv / .png files and does NOT touch or overwrite any existing
# Phase 10 (or other phase) results. Nothing prior is modified.
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

## Configuration — Phase 10 verbatim, plus the `ordered` regime target

In [ ]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; mlp_mult:int=2
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=20
    eval_n:int=1500; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    # factorial axes
    inputs:tuple=("trained","random")   # <-- the new controlled factor
    chaos_regimes:tuple=("ordered","edge","deep")
    tasks:tuple=("copy","forecast")

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=40; CFG.calib_steps=40; CFG.calib_warm=10
    print("*** SMOKE TEST ***")

def build_conditions(cfg):
    conds=[]
    for inp in cfg.inputs:
        for reg in cfg.chaos_regimes:
            for task in cfg.tasks:
                conds.append(dict(arm="chaos",regime=reg,input=inp,task=task))
        for task in cfg.tasks:
            conds.append(dict(arm="linear",regime="edge",input=inp,task=task))
    # GRU reference: fully trained (input trained by definition), both tasks
    for task in cfg.tasks:
        conds.append(dict(arm="gru",regime="edge",input="trained",task=task))
    return conds
CONDS=build_conditions(CFG)
print(f"conditions={len(CONDS)}  seeds={CFG.seeds}  total runs={len(CONDS)*CFG.seeds}")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

## Cells: chaos / linear / GRU (verbatim from Phase 10)

The `ChaosCell` is unchanged: the new `ordered` regime is realized purely through gamma-calibration to a negative target, using the identical machinery as `edge` and `deep`. No new cell type is introduced.

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass
    def freeze_input(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"])

## gamma-calibration (verbatim from Phase 10)

The sanity check below now also calibrates the `ordered` target, confirming the three regimes are reachable on the same cell.

In [ ]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.05,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

# sanity: verify regimes calibrate
for reg in CFG.chaos_regimes:
    c=ChaosCell(CFG,reg).to(DEVICE); gam,lam=calibrate_gamma(c,CFG,reg,seed=0)
    print(f"regime={reg:5s} target={CFG.lambda_target[reg]:+.2f} -> gamma={gam:.3f} measured lambda={lam:+.3f}")

## Model with freeze_input control (verbatim from Phase 10)

In [ ]:
class SeqModel(nn.Module):
    def __init__(self,cfg,cond,in_kind,in_dim,out_dim):
        super().__init__(); self.cfg=cfg; self.cond=cond; self.in_kind=in_kind
        if in_kind=="tokens": self.emb=nn.Embedding(in_dim,cfg.d_model)
        else: self.inp=nn.Linear(1,cfg.d_model)
        self.cell=make_cell(cfg,cond)
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        self.achieved_lambda=float("nan")
        # reservoir arms (chaos, linear) freeze the recurrence; GRU trains fully
        if cond["arm"] in ("chaos","linear"): self.cell.freeze_recurrence()
        # canonical RC: also freeze the input projection AND the encoder
        if cond["input"]=="random" and cond["arm"] in ("chaos","linear"):
            self.cell.freeze_input()
            if in_kind=="tokens":
                self.emb.weight.requires_grad_(False)
            else:
                for p_ in self.inp.parameters(): p_.requires_grad_(False)
    def calibrate(self,cfg):
        if self.cond["arm"]=="chaos":
            _,self.achieved_lambda=calibrate_gamma(self.cell,cfg,self.cond["regime"],
                                                   seed=abs(hash(self.cond["regime"]))%1000  # NOTE: Python string hashing is salted per process (PYTHONHASHSEED); set PYTHONHASHSEED=0 for bit-exact reproducibility across sessions)
        return self
    def encode(self,x):
        return self.emb(x) if self.in_kind=="tokens" else self.inp(x.unsqueeze(-1))
    def forward(self,x):
        e=self.encode(x); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

## Tasks: copy + forecast (verbatim from Phase 10)

In [ ]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
CHANCE=dict(copy=1.0/V, forecast=1.0/CFG.fc_bins)
def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

## Trainer (verbatim from Phase 10)

In [ ]:
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def run_condition(cond,seed,cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    inp,tgt,in_kind,in_dim,out_dim=TASK_FN[cond["task"]](cfg,4,DEVICE,gen)
    model=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE); model.calibrate(cfg)
    evalset=build_eval(cfg,cond["task"])
    params=[p for p in model.parameters() if p.requires_grad]
    n_train=sum(p.numel() for p in params)
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    div=False
    for step in range(cfg.steps):
        inp,tgt,_,_,_=TASK_FN[cond["task"]](cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc(model,evalset) if not div else float("nan")
    return dict(**cond,seed=seed,diverged=div,acc=acc,n_trainable=n_train,
                achieved_lambda=round(float(model.achieved_lambda),3))

## NEW 1 — Conditional (input-driven) Lyapunov exponent on the forecast task

The Phase 10 `map_lyapunov` measures the *autonomous* map exponent (no input). The reviewer asks for the
exponent **on the forecast task**, i.e. the *conditional* exponent along the actual driven trajectory.
We implement it correctly here: we propagate an orthonormal frame through the per-step Jacobian of the
full driven reservoir update h_{t+1} = fold(phi(h_t) + B x_t), evaluated along the real forecast input
sequence, re-orthonormalizing each step (Benettin/QR). This is the driven analogue of the autonomous
estimator and is the quantity whose sign tells us whether the driven reservoir expands or contracts.

We report it for the frozen **edge** reservoir under two input conditions:
- random B (canonical protocol): expected to remain near zero from below (chaos preserved);
- trained B: expected to be driven negative (input-driven synchronization suppresses the chaos),
reproducing on forecast the mechanism the paper establishes on copy.

In [ ]:
@torch.no_grad()
def conditional_lyapunov_forecast(model, cfg, seed=0, n_warm=100, n_acc=400):
    """Driven (conditional) maximal Lyapunov exponent of the frozen reservoir along a real
    forecast input sequence. Works for a ChaosCell reservoir (chaos arm). Uses the SeqModel's
    own encoder (self.inp) so the drive matches training exactly."""
    model.eval()
    cell = model.cell
    assert hasattr(cell, "W_mix"), "conditional-lambda is defined for the chaos reservoir cell"
    dev = cell.W_mix.device; N = cfg.state_dim
    g = torch.Generator(device=dev); g.manual_seed(seed)
    # generate one real forecast input stream (logistic r=3.9), same generator discipline as the task
    L = n_warm + n_acc + 1; r = 3.9
    x0 = 0.1 + 0.8*torch.rand(1,1,device=dev,generator=g); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X = torch.cat(xs,1)[0]                         # (L+1,)
    drive = model.encode(X.view(1,-1)).squeeze(0)  # (L+1, d_model) through the model's own encoder
    h = cell.init_state(1, dev, torch.float32).squeeze(0)  # (N,)
    Q = torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum = torch.zeros(N, device=dev)
    def driven_step(z, xvec):
        # exact SeqModel step: h <- fold(phi(h) + x @ B^T)
        return triangle_fold(cell.phi(z) + xvec @ cell.B.T)
    for t in range(n_warm + n_acc):
        xvec = drive[t]
        J = torch.autograd.functional.jacobian(lambda z: driven_step(z, xvec), h)
        h = driven_step(h, xvec)
        Q, R = torch.linalg.qr(J @ Q)
        d = torch.sign(torch.diagonal(R)); Q = Q*d; R = R*d.unsqueeze(0)
        if t >= n_warm:
            logsum = logsum + torch.log(torch.abs(torch.diagonal(R)) + 1e-12)
    return float(torch.sort(logsum/n_acc, descending=True)[0][0])


## NEW 2 — Linear memory capacity (MC) and separation on the frozen reservoir

**Memory capacity (Jaeger 2001).** Drive the frozen reservoir with an i.i.d. uniform scalar input
stream u_t. Collect reservoir states h_t. For each delay k, fit a linear readout predicting u_{t-k}
from h_t and record the squared correlation r^2(k); MC = sum_k r^2(k). MC quantifies how much past-input
information the frozen reservoir linearly retains — the canonical RC property that edge-of-chaos
reservoirs are theorized to maximize. No training of the reservoir is involved (pure linear algebra on
frozen states), so MC is a property of the dynamical regime alone.

**Separation (kernel quality).** Drive the reservoir with many independent random input streams and
measure the effective dimensionality of the resulting state matrix via the participation ratio of its
singular values, (sum s_i)^2 / sum s_i^2, a standard separation proxy. Higher separation = richer, more linearly-separable
representation of distinct input histories — one of the intermediate computational properties Reviewer 1
asks us to characterize.

In [ ]:
@torch.no_grad()
def memory_capacity(cell, cfg, encode_fn, seed=0, T=1500, washout=200, max_delay=40):
    """Linear short-term memory capacity of a frozen reservoir cell driven by i.i.d. uniform input."""
    dev = next(p.device for p in cell.parameters()); N = cfg.state_dim
    g = torch.Generator(device=dev); g.manual_seed(seed)
    u = torch.rand(T, generator=g, device=dev)          # uniform [0,1]
    drive = encode_fn(u.view(1,-1)).squeeze(0)           # (T, d_model) via model encoder
    h = cell.init_state(1, dev, torch.float32).squeeze(0)
    H=[]
    for t in range(T):
        h = triangle_fold(cell.phi(h) + drive[t] @ cell.B.T) if hasattr(cell,"W_mix") \
            else torch.tanh(h @ cell.W_lin.T + drive[t] @ cell.B.T)
        H.append(h)
    H = torch.stack(H)                                   # (T, N)
    Hd = H[washout:]; um = u[washout:]
    # add bias column
    X = torch.cat([Hd, torch.ones(Hd.shape[0],1,device=dev)],1)
    mc = 0.0; per=[]
    for k in range(1, max_delay+1):
        if k >= Hd.shape[0]-2: break
        Xk = X[k:]; yk = um[:-k]                          # predict u_{t-k} from h_t
        # ridge-regularized least squares, solved robustly (lstsq tolerates rank deficiency)
        lam_ridge = 1e-3
        A = Xk.T @ Xk + lam_ridge*torch.eye(Xk.shape[1],device=dev)
        b = Xk.T @ yk
        w = torch.linalg.lstsq(A, b.unsqueeze(1)).solution.squeeze(1)
        pred = Xk @ w
        # squared correlation
        vp = pred - pred.mean(); vy = yk - yk.mean()
        r2 = (vp@vy)**2 / ((vp@vp)*(vy@vy) + 1e-12)
        r2 = float(torch.clamp(r2,0,1)); per.append(r2); mc += r2
    return mc, per

@torch.no_grad()
def separation(cell, cfg, encode_fn, seed=0, n_streams=200, T=60):
    """Participation-ratio effective rank, (sum s_i)^2 / sum s_i^2, of the final reservoir
    states collected across many independent random input streams. Reported as 'separation'."""
    dev = next(p.device for p in cell.parameters()); N = cfg.state_dim
    g = torch.Generator(device=dev); g.manual_seed(seed)
    finals=[]
    for s in range(n_streams):
        u = torch.rand(T, generator=g, device=dev)
        drive = encode_fn(u.view(1,-1)).squeeze(0)
        h = cell.init_state(1, dev, torch.float32).squeeze(0)
        for t in range(T):
            h = triangle_fold(cell.phi(h) + drive[t] @ cell.B.T) if hasattr(cell,"W_mix") \
                else torch.tanh(h @ cell.W_lin.T + drive[t] @ cell.B.T)
        finals.append(h)
    M = torch.stack(finals)                               # (n_streams, N)
    M = M - M.mean(0, keepdim=True)
    sv = torch.linalg.svdvals(M)
    # effective rank (participation ratio of singular values) — normalized separation proxy
    eff_rank = float((sv.sum()**2) / (sv.pow(2).sum() + 1e-12))
    return eff_rank


## NEW 3 — Run the regime-pure reservoir comparison (forecast, canonical random-input, logistic)

This is the load-bearing run: ordered / edge / deep frozen reservoirs vs the linear reservoir, under the
canonical readout-only protocol, on forecast, at the full seed count. For each chaos regime and seed we
also record MC, separation, and (for the edge arm) the conditional forecast-lambda under random-B and
trained-B.

In [ ]:
def run_regime_sweep(cfg):
    rows=[]; diag=[]
    # forecast, canonical random-input: ordered / edge / deep chaos, plus linear reference
    for reg in cfg.chaos_regimes:
        for sd in range(cfg.seeds):
            cond=dict(arm="chaos",regime=reg,input="random",task="forecast")
            r=run_condition(cond,sd,cfg); rows.append(r)
            # diagnostics on the SAME frozen, calibrated reservoir:
            torch.manual_seed(sd); np.random.seed(sd)
            inp,tgt,in_kind,in_dim,out_dim=TASK_FN["forecast"](cfg,4,DEVICE,
                torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu").manual_seed(sd))
            m=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE); m.calibrate(cfg)
            mc,per=memory_capacity(m.cell,cfg,m.encode,seed=sd)
            sep=separation(m.cell,cfg,m.encode,seed=sd)
            diag.append(dict(regime=reg,seed=sd,achieved_lambda=m.achieved_lambda,
                             memory_capacity=round(mc,4),separation=round(sep,4),acc=r["acc"]))
            print(f"  chaos/{reg:7s} s={sd} acc={r['acc']:.4f} MC={mc:.3f} sep={sep:.3f} lam={m.achieved_lambda:+.3f}")
    # linear reference (forecast, random-input)
    for sd in range(cfg.seeds):
        cond=dict(arm="linear",regime="edge",input="random",task="forecast")
        r=run_condition(cond,sd,cfg); rows.append(r)
        torch.manual_seed(sd); np.random.seed(sd)
        m=SeqModel(cfg,cond,"scalar",1,cfg.fc_bins).to(DEVICE)
        mc,per=memory_capacity(m.cell,cfg,m.encode,seed=sd)
        sep=separation(m.cell,cfg,m.encode,seed=sd)
        diag.append(dict(regime="linear",seed=sd,achieved_lambda=float("nan"),
                         memory_capacity=round(mc,4),separation=round(sep,4),acc=r["acc"]))
        print(f"  linear        s={sd} acc={r['acc']:.4f} MC={mc:.3f} sep={sep:.3f}")
    return pd.DataFrame(rows), pd.DataFrame(diag)

t0=time.time()
RES, DIAG = run_regime_sweep(CFG)
print(f"\nregime sweep wall clock: {(time.time()-t0)/60:.1f} min")
savetable(RES,"phase14_regime_sweep_acc.csv")
savetable(DIAG,"phase14_regime_diagnostics.csv")


## NEW 4 — Conditional forecast-lambda: edge reservoir, random-B vs trained-B

In [ ]:
cond_rows=[]
for inp in ("random","trained"):
    for sd in range(CFG.seeds):
        cond=dict(arm="chaos",regime="edge",input=inp,task="forecast")
        torch.manual_seed(sd); np.random.seed(sd)
        m=SeqModel(CFG,cond,"scalar",1,CFG.fc_bins).to(DEVICE); m.calibrate(CFG)
        # for trained-B we must actually train B+readout first, so the measured lambda reflects the
        # trained input projection; for random-B the reservoir is already in canonical frozen form.
        if inp=="trained":
            gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu").manual_seed(sd)
            params=[p for p in m.parameters() if p.requires_grad]
            opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
            sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps))
            for step in range(CFG.steps):
                inp_t,tgt,_,_,_=TASK_FN["forecast"](CFG,CFG.batch,DEVICE,gen)
                lg=m(inp_t); Lz=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
                opt.zero_grad(set_to_none=True); Lz.backward()
                torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip); opt.step(); sched.step()
        lam_cond=conditional_lyapunov_forecast(m,CFG,seed=sd)
        cond_rows.append(dict(input=inp,seed=sd,conditional_lambda_forecast=round(lam_cond,4),
                              achieved_autonomous_lambda=round(m.achieved_lambda,4)))
        print(f"  edge/in={inp:7s} s={sd} conditional_lambda_forecast={lam_cond:+.4f}")
CONDLAM=pd.DataFrame(cond_rows); savetable(CONDLAM,"phase14_conditional_lambda_forecast.csv")
print("\nMean conditional forecast-lambda by input condition:")
print(CONDLAM.groupby("input").conditional_lambda_forecast.agg(["mean","std"]))


## Analysis + summary table

In [ ]:
import numpy as np
def ci95(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(0); b=rng.choice(x,size=(10000,len(x)),replace=True).mean(1)
    return float(np.percentile(b,2.5)), float(np.percentile(b,97.5))

print("="*72)
print("REGIME-PURE RESERVOIR COMPARISON (forecast, canonical random-input, logistic)")
print("="*72)
summ=[]
for reg in list(CFG.chaos_regimes)+["linear"]:
    d=DIAG[DIAG.regime==reg]
    acc=d.acc.values; mc=d.memory_capacity.values; sep=d.separation.values
    lo,hi=ci95(acc)
    summ.append(dict(regime=reg,
                     mean_acc=round(np.nanmean(acc),4),acc_ci=f"[{lo:.3f},{hi:.3f}]",
                     mean_MC=round(np.nanmean(mc),3),mean_sep=round(np.nanmean(sep),3),
                     mean_lambda=round(np.nanmean(d.achieved_lambda.values),3)))
SUMM=pd.DataFrame(summ); print(SUMM.to_string(index=False)); savetable(SUMM,"phase14_summary.csv")

# verdict logic for R1#1: does edge beat BOTH ordered and linear, with CI separation?
edge_acc=DIAG[DIAG.regime=="edge"].acc.values
ord_acc =DIAG[DIAG.regime=="ordered"].acc.values
lin_acc =DIAG[DIAG.regime=="linear"].acc.values
def better(a,b):
    d=np.asarray(a)-np.asarray(b)[:len(a)] if len(a)==len(b) else None
    lo,hi=ci95(np.asarray(a)) ; lo2,hi2=ci95(np.asarray(b))
    return np.nanmean(a)>np.nanmean(b)
print("\nR1#1 regime-purity check:")
print(f"  edge mean={np.nanmean(edge_acc):.3f}  ordered mean={np.nanmean(ord_acc):.3f}  linear mean={np.nanmean(lin_acc):.3f}")
print(f"  edge > ordered: {np.nanmean(edge_acc)>np.nanmean(ord_acc)}   edge > linear: {np.nanmean(edge_acc)>np.nanmean(lin_acc)}")
print("  -> edge exceeding BOTH would indicate a criticality-specific benefit; edge ~= ordered >> deep\n     instead indicates a nonlinear-reservoir benefit that is not specific to criticality.")


## Figure — three-point regime axis: accuracy, memory capacity, separation (300 DPI)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.4))
order=["ordered","edge","deep"]
xlab=[f"ordered\n(λ≈-0.30)","edge\n(λ≈0⁻)","deep\n(λ≈+0.30)"]
def series(metric):
    return [np.nanmean(DIAG[DIAG.regime==r][metric].values) for r in order], \
           [np.nanstd(DIAG[DIAG.regime==r][metric].values) for r in order]
lin_acc=np.nanmean(DIAG[DIAG.regime=="linear"].acc.values)
lin_mc =np.nanmean(DIAG[DIAG.regime=="linear"].memory_capacity.values)
lin_sep=np.nanmean(DIAG[DIAG.regime=="linear"].separation.values)
for ax,(metric,title,linref) in zip(axes,[("acc","Forecast accuracy",lin_acc),
                                           ("memory_capacity","Memory capacity (MC)",lin_mc),
                                           ("separation","Separation (eff. rank)",lin_sep)]):
    m,s=series(metric)
    ax.bar(range(3),m,yerr=s,capsize=4,color=["#4C72B0","#2E7D32","#B71C1C"],edgecolor="white")
    ax.axhline(linref,ls="--",color="#777",lw=1.3,label="linear reservoir")
    ax.set_xticks(range(3)); ax.set_xticklabels(xlab,fontsize=10)
    ax.set_title(title,fontsize=13); ax.grid(alpha=0.25,axis="y"); ax.legend(fontsize=9)
fig.suptitle("Regime-pure reservoir controls: ordered and edge match on accuracy; deep chaos does not help",fontsize=13)
fig.tight_layout(); savefig(fig,"phase14_regime_axis.png"); plt.close(fig)
print("figure saved")
